In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP04 - TP Sole Source
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   5. Single Source File: hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
   
   BUSINESS QUESTION: 
   Number of active third party vendors who are 1. single source vendors AND 2. 
   located in high risk jurisdictions.
   
   LOGIC SUMMARY (Updated with Flattening & Bridging Architecture):
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
   2. SINGLE SOURCE BRIDGE: Finds 'Single Source' in the EngagementNumber column of the 
      contracts table, extracting the true ID from the ThirdPartyName column.
   3. HIGH RISK EVALUATION: Checks Col D, M, N against the ABAC High Risk list. If all 
      three are blank, automatically defaults to High Risk.
   4. EXCEPTION HANDLING: Uses a 'Placeholder Swap' ([COMMA]) to protect known unit 
      names that contain commas (e.g., 'CAD PB - RESL...') from being improperly split.
   5. FLATTEN LOBs: Columns K and L can contain comma-separated lists. Uses LATERAL 
      VIEW explode(split(...)) to expand these strings into individual evaluation rows.
   6. RESTORE LOBs: Swaps '[COMMA]' back to a real comma after the explosion.
   7. TPRM STRING MAPPING & NAME BRIDGE: Maps expanded LOBs to AU IDs using SAFE LIKE 
      syntax. Then bridges the Mapping Table's String Name back to the Master List's 
      Numeric BusinessID.
   8. DEDUPLICATION: Uses COUNT(DISTINCT EngagementNumber) to ensure 1 engagement = 1 count.
   9. FINAL OUTPUT: Strict 6-column structure, converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- Grab valid TPRM Mapping Strings
    -- Note: Assessable_Unit_ID actually contains the string Name, not the numeric ID
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

High_Risk_Countries AS (
    -- Grab high-risk jurisdictions from the ABAC rating table
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    -- 2. Bridge the true Engagement ID from the Single Source file
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3 & 4. Extract TP data, evaluate missing jurisdictions, and apply Exception Dictionary
    SELECT 
        p.EngagementNumber,
        
        -- EXCEPTION DICTIONARY: Protect commas in known LOBs using '[COMMA]'
        REPLACE(p.owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_owning_lob,
        REPLACE(p.lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_lob_using,
        
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    -- INNER JOIN to strictly keep only Single Source engagements
    INNER JOIN Single_Source_Engagements s
        ON TRIM(p.EngagementNumber) = s.True_Engagement_ID
),

High_Risk_Engagements AS (
    -- Filter strictly for High Risk countries or Missing Jurisdictions
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.safe_owning_lob,
        tp.safe_lob_using
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

Flattened_LOBs AS (
    -- 5 & 6. FLATTEN & RESTORE RULE: Split by commas, explode, then restore the protected commas
    SELECT 
        EngagementNumber, 
        -- RESTORE: Put the actual commas back so it matches the Mapping Table!
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM High_Risk_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 7 & 8. Map engagements, bridge to Master ID, and count distinct engagements
    SELECT 
        mast.BusinessID,
        COUNT(DISTINCT f.EngagementNumber) AS High_Risk_Count
    FROM Flattened_LOBs f
    
    -- Safe Mapping Join
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
        
    -- Name Bridge Join
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_AU_Name)) = UPPER(TRIM(mast.AU_Name))
    GROUP BY mast.BusinessID
)

-- 9. Final Output: Strict 6-column structure anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP04' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP04 - Sole Source Traceability with Comma Exceptions
   
   PURPOSE: Verifies the Single Source bridge logic, Risk Flag logic, shows how the 
   Exception Dictionary safely protected comma-containing LOBs during the flattening 
   process, and displays the Master AU lookup status.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Filtered_Engagements AS (
    -- Bridge to Single Source, evaluate High Risk, and prep for flattening
    SELECT 
        p.EngagementNumber,
        p.location_of_tp,
        p.country_prod_serv_originates,
        p.Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(p.location_of_tp), '') = ''
              AND COALESCE(TRIM(p.country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '' THEN 'Yes - All Columns Blank'
             ELSE 'No' END AS Missing_Jurisdiction_Flag,
        
        -- The original untouched strings
        p.owning_lob AS Full_Col_K,
        p.lob_using AS Full_Col_L,
        
        -- Exception Dictionary applied
        REPLACE(p.owning_lob, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_owning_lob,
        REPLACE(p.lob_using, 'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing') AS safe_lob_using
        
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 p
    INNER JOIN Single_Source_Engagements s
        ON TRIM(p.EngagementNumber) = s.True_Engagement_ID
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(p.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(p.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(p.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR (COALESCE(TRIM(p.location_of_tp), '') = '' AND COALESCE(TRIM(p.country_prod_serv_originates), '') = '' AND COALESCE(TRIM(p.Td_Countries_Impacted), '') = '')
),

Flattened AS (
    SELECT 
        f.EngagementNumber,
        f.location_of_tp AS Col_D,
        f.country_prod_serv_originates AS Col_M,
        f.Td_Countries_Impacted AS Col_N,
        f.Missing_Jurisdiction_Flag,
        f.Full_Col_K,
        f.Full_Col_L,
        TRIM(exploded_lob) AS Raw_Exploded_Chunk,
        
        -- The restored chunk
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Restored_Expanded_LOB
        
    FROM Filtered_Engagements f
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

SELECT DISTINCT
    f.EngagementNumber,
    'Yes' AS Confirmed_Single_Source,
    
    -- Risk Traceability
    f.Col_D,
    f.Col_M,
    f.Col_N,
    f.Missing_Jurisdiction_Flag,
    
    -- LOB Flattening Traceability
    f.Full_Col_K,
    f.Full_Col_L,
    f.Restored_Expanded_LOB,
    
    -- Mapping & Bridging Traceability
    map.TPRM_Units AS Matched_Mapping_String,
    map.Assessable_Unit_ID AS Mapping_Table_AU_Name,
    mast.Master_Numeric_ID AS Bridged_Master_ID,
    CASE WHEN mast.Master_Numeric_ID IS NULL THEN 'MISSING FROM MASTER LIST' ELSE 'SUCCESSFULLY BRIDGED' END AS Master_List_Status
    
FROM Flattened f
-- Safe Mapping Join using the RESTORED string
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map 
    ON UPPER(f.Restored_Expanded_LOB) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
-- Name Bridge Join
LEFT JOIN Master_AUs mast 
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(TRIM(mast.Master_AU_Name))
ORDER BY f.EngagementNumber;